In [ ]:
import pathlib

import ampworks as amp

extension = ['.csv', '.txt', '.xls', '.xlsx']


def test_read_csv_custom_aliases(extension):
    path = pathlib.Path('.')
    file = path / 'dummy_data' / ('aliases' + extension)

    print(file)

    all_readers = {
        '.csv': amp.read_csv,
        '.txt': amp.read_table,
        '.xls': amp.read_excel,
        '.xlsx': amp.read_excel,
    }

    reader = all_readers[extension]

    aliases = amp.HeaderAliases(
        Seconds='elapsed_s',
        Amps='amps_raw',
        Volts='volts_raw',
    )

    data = reader(file, aliases=aliases, extra_columns={'Meta': None})

    # assert set(['Seconds', 'Amps', 'Volts', 'Meta']).issubset(data.columns)
    # assert data['Meta'].to_list() == ['a', 'b']

    return data

In [ ]:
for ext in extension:
    data = test_read_csv_custom_aliases(ext)
    print(data.dtypes)

In [ ]:
# Remove unnecessary characters from header strings
def strip_chars(string: str | list[str] | None) -> str | list[str] | None:
    if string is None:
        return None
    elif isinstance(string, list):
        return [strip_chars(s) for s in string]

    transmap = str.maketrans('(/,', '...', ' _-#<>)')
    return string.lower().translate(transmap)


# Matches input headers with aliases of the standard headers
def header_matches(
    headers: list[str], targets: list[str], aliases: amp.HeaderAliases,
) -> bool:
    headers = strip_chars(headers)

    checks = {}
    for k in targets:
        if any(alias in headers for alias in aliases[k]):
            checks[k] = True
        else:
            checks[k] = False

    return all(checks.values())

In [ ]:
import csv

import polars as pl

from ampworks._checks import _check_type

file = 'dummy_data/aliases.csv'

REQUIRED_HEADERS = ['Seconds', 'Amps', 'Volts']

aliases = amp.HeaderAliases(
    Seconds='elapsed_s',
    Amps='amps_raw',
    Volts='volts_raw',
)

if aliases is None:
    aliases = amp.HeaderAliases()

_check_type('aliases', aliases, amp.HeaderAliases)

options = {'separator': ',', 'skip_rows': 0, 'ignore_errors': True}
with open(file, encoding='latin1') as datafile:
    reader = csv.reader(datafile, delimiter=',')

    found_header = False
    for idx, line in enumerate(reader):
        print(line)
        if header_matches(line, REQUIRED_HEADERS, aliases):
            options['skip_rows'] = idx
            found_header = True
            break

    if found_header:
        df = pl.read_csv(file, **options).to_pandas()
        # return standardize_headers(df, aliases, extra_columns)

# return Dataset()

In [ ]:
# Matches input headers with aliases of the standard headers
def header_matches(
    headers: list[str], targets: list[str], aliases: amp.HeaderAliases,
) -> bool:
    headers = strip_chars(headers)
    print(headers)
    print(targets)
    for k in targets:
        print(aliases[k])

    checks = {}
    for k in targets:
        if any(alias in headers for alias in aliases[k]):
            checks[k] = True
        else:
            checks[k] = False

    print(checks)
    return all(checks.values())


line = ['elapsed_s', 'amps_raw', 'volts_raw', 'Meta']
header_matches(line, REQUIRED_HEADERS, aliases)

In [ ]:
aliases

In [ ]:
amp.read_csv('dummy_data/aliases.csv')